# ACE-Net Preprocessor — TEST SET [TRACKS_MELD]
### Target Evaluation Split: **TEST SET** (TRACKS_MELD)
### Output Target: `Google Drive > THESIS_MOTHERFILE > baseline_training > test > TRACKS_MELD > shards > shard_0001`

## Step 1: Connect to GPU & Mount Google Drive

In [ ]:
from google.colab import drive
import os, sys, torch

drive.mount('/content/drive')
print('GPU Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

## Step 2: Clone Repository & Checkout Branch

In [ ]:
%cd /content
!rm -rf Baseline_Training
!git clone https://github.com/gjvlio/Baseline_Training.git
%cd Baseline_Training
!git checkout feat/baseline-preprocessing-jc
!git pull
!git log --oneline -1

## Step 3: Install Required Dependencies

In [ ]:
!pip install -q --no-deps facenet-pytorch
!pip install -q --no-deps git+https://github.com/openai/whisper.git
print('Dependencies installed successfully!')

## Step 4: Fast Unzip Assigned Datasets (tracks_1_2_3_4.zip, meld_raw.zip) to Local SSD

In [ ]:
import os
LOCAL_RAW = '/content/data/raw'
os.makedirs(LOCAL_RAW, exist_ok=True)

print('Fast unzipping ['tracks_1_2_3_4.zip', 'meld_raw.zip'] to local SSD (/content/data/raw)...')
!unzip -q -n '/content/drive/MyDrive/THESIS_MOTHERFILE/datasets/tracks_1_2_3_4.zip' -d '/content/data/raw'
!unzip -q -n '/content/drive/MyDrive/THESIS_MOTHERFILE/datasets/meld_raw.zip' -d '/content/data/raw'
print('Unzip complete! Local files ready.')

## Step 5: Execute Preprocessing for `test` [TRACKS_MELD]

In [ ]:
!python scripts/preprocess/run_shard.py \
    --account 'eval_runner@gmail.com' \
    --dataset 'test/TRACKS_MELD' \
    --shard '0001' \
    --manifest '/content/Baseline_Training/data/manifests/eval_shards/test/TRACKS_MELD_manifest.csv' \
    --raw_dir '/content/data/raw' \
    --drive_root '/content/drive/MyDrive/THESIS_MOTHERFILE/baseline_training' \
    --device cuda

## Step 6: Post-Run Integrity Check & Verification

In [ ]:
import json, glob
from pathlib import Path

ckpt_file = Path('/content/drive/MyDrive/THESIS_MOTHERFILE/baseline_training/test/TRACKS_MELD/checkpoints/shard_0001_checkpoint.json')
out_shard = Path('/content/drive/MyDrive/THESIS_MOTHERFILE/baseline_training/test/TRACKS_MELD/shards/shard_0001')

if ckpt_file.exists():
    with open(ckpt_file) as f:
        data = json.load(f)
    print('=' * 60)
    print('SHARD STATUS:', data.get('status'))
    print('Completed Clips:', len(data.get('completed_ids', [])))
    print('Failed Clips   :', len(data.get('failed_ids', [])))
    print('=' * 60)
else:
    print('Checkpoint not found. Run Step 5 first.')